# PW3: Mice's sleep stages classification with MLP

Group:

- David Schildböck
- Kénan Augsburger

Goals:

- Prepare and pre-process the available data to train a neural-network for solving a classification task.
- Design experiments to exploit the available data to come up with an automatic classification system.
- Design, train and evaluate neural networks (Multi-Layer Perceptron) to solve a real-world problem.
- Perform hyper-parameter tuning to find an appropriate neural network architecture and to achieve a good performing model.


## Introduction

In [2]:
# Basic imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from tqdm.notebook import tqdm


# Setting up the visualisation style
sns.set_theme(style="whitegrid")

### Training data:

- EEG_mouse_data_1.csv (same mouse as PW1)
- EEG_mouse_data_2.csv (another mouse for training)

Note: if you want to concatenate pandas dataframes, you can do it with « pd.concat ».

In [19]:
raw_train_data_1 = pd.read_csv("pw3_data/EEG_mouse_data_1.csv")
raw_train_data_2 = pd.read_csv("pw3_data/EEG_mouse_data_2.csv")
raw_train_data = pd.concat([raw_train_data_1, raw_train_data_2], ignore_index=True)

### Testing data:

- EEG_mouse_data_test.csv (another mouse for testing)

All of these mice are genetical clones of each other so the data will be slightly different from
mouse to mouse but will still be similar.  
We don't provide any notebook. You can use code of previous practical works to help you. You
can choose whether you want to do it on Colab or not.


In [17]:
raw_test_data = pd.read_csv("pw3_data/EEG_mouse_data_test.csv")

### Preprocessing:

- Choose 25 features
- Normalize data

see: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html


In [18]:
# Exploring the datasets

raw_test_data.describe()

,amplitude_around_1_Hertz,amplitude_around_2_Hertz,amplitude_around_3_Hertz,amplitude_around_4_Hertz,amplitude_around_5_Hertz,amplitude_around_6_Hertz,amplitude_around_7_Hertz,amplitude_around_8_Hertz,amplitude_around_9_Hertz,amplitude_around_10_Hertz,...,amplitude_around_92_Hertz,amplitude_around_93_Hertz,amplitude_around_94_Hertz,amplitude_around_95_Hertz,amplitude_around_96_Hertz,amplitude_around_97_Hertz,amplitude_around_98_Hertz,amplitude_around_99_Hertz,amplitude_around_100_Hertz,amplitude_around_101_Hertz
count,80109.000000,80109.000000,80109.000000,80109.000000,80109.000000,80109.000000,80109.000000,80109.000000,80109.000000,80109.000000,...,8.010900e+04,8.010900e+04,8.010900e+04,8.010900e+04,8.010900e+04,8.010900e+04,8.010900e+04,8.010900e+04,8.010900e+04,8.010900e+04
mean,-0.000023,0.000015,0.000016,0.000016,0.000014,0.000012,0.000013,0.000013,0.000010,0.000007,...,9.927835e-09,7.624872e-09,5.687167e-09,4.156366e-09,3.009007e-09,2.124984e-09,1.519217e-09,1.131444e-09,1.037376e-09,-1.997249e-04
std,0.003534,0.000018,0.000018,0.000016,0.000014,0.000011,0.000012,0.000012,0.000009,0.000007,...,8.602050e-09,6.664576e-09,4.960118e-09,3.668513e-09,2.606265e-09,1.796833e-09,1.273646e-09,9.540812e-10,8.455716e-10,1.413119e-02
min,-0.250000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.000000e+00
25%,0.000005,0.000005,0.000005,0.000005,0.000005,0.000004,0.000005,0.000005,0.000004,0.000003,...,4.278843e-09,3.244971e-09,2.438945e-09,1.796409e-09,1.308106e-09,9.408982e-10,6.787478e-10,5.071618e-10,4.835167e-10,2.897480e-10
50%,0.000012,0.000009,0.000010,0.000010,0.000009,0.000008,0.000009,0.000009,0.000007,0.000005,...,7.447277e-09,5.702682e-09,4.262453e-09,3.121290e-09,2.265915e-09,1.610378e-09,1.155194e-09,8.629032e-10,8.128287e-10,1.319130e-09
75%,0.000026,0.000018,0.000020,0.000020,0.000018,0.000015,0.000016,0.000016,0.000013,0.000009,...,1.275024e-08,9.848925e-09,7.301070e-09,5.344233e-09,3.873223e-09,2.741333e-09,1.956633e-09,1.452218e-09,1.331247e-09,3.895100e-09
max,0.007021,0.000375,0.000326,0.000272,0.000245,0.000167,0.000167,0.000215,0.000150,0.000101,...,1.511856e-07,1.433177e-07,9.574633e-08,1.375192e-07,5.607566e-08,4.832078e-08,3.697477e-08,3.940120e-08,2.649568e-08,2.951280e-07


Without the sleep state label, it's difficult to plot the frequencies for each
given states and get an idea of which features are the best for classification.

When looking online, it seems that it's best to have dat from the frequency bands
- $\delta$: 0.1-4Hz
- $\theta$: 4-10Hz
- $\alpha$: 10-15Hz
- $\beta$: 15-30Hz
- $\gamma$: 30-100Hz

In [21]:
features = [*range(1,16), *range(21,26), 59, 69, 79, 89, 99]

train_data = raw_train_data.iloc[:, features]
test_data = raw_test_data.iloc[:, features]

### Validation:

- Use 3-fold cross-validation and shuffle data

**Note**: we shuffle the data but it's not a perfect solution. In that way, with two data points that
are few seconds apart, one could be in training set and the other one in validation, which
means that the validation will be highly correlated with the training and would not represent
reliably the model's performance on unknown data. If we split data without shuffling, we
would have folds with nearly only awake state which would not be desirable as well.


### Training history: 
- Plot training and validation loss 

### Performance metrics: 
- Confusion matrix 
- F1-score (of each class and 'micro', there is a parameter for « micro », check sklearn) 

## 2. Report 

Don't include answers to « Questions to ask yourselves » in your report. 

For each experiment and for the competition, include your model summary, your performances 
results, your training history plot, and analyse your results (Is there overfitting? Should we train 
for longer? What can we say about the confusions the model makes? How did you choose the 
learning rate? How did you choose the number of neurons and number of layers? …) 

For the competition part, explain briefly one of the idea you chose and if you did some other 
modifications, just state them in your report. 

## 3. First experiment: awake / asleep 

Here we will classify between only two states: « awake » and « asleep ». 
You must group n-rem and rem sleep stages into a class « asleep ».  

*Questions to ask yourselves:*

- Is it correct to normalize before splitting before training and validation? Why? 

## 4. Second experiment: awake / n-rem / rem 

Classify between the three states « awake », « n-rem » and « rem ».  

*Questions to ask yourselves:*

- What do you need to change compared to the previous experiment? 
- How many outputs neurons should we have? 
- How to encode the model's output when there's more than two classes? (take a look at one-hot encoding) 
- Is the target encoding different if you use « tanh » instead of  « sigmoid » for the outputs activation function? 
- How to decide which class is predicted? 
- If you have « tanh » or « sigmoid » as the output activation function and if you decide which class is predicted with a threshold (e.g. > 0 or > 0.5): 
    - Is it possible: 
        - That multiple class are predicted 
        - That no class is predicted 
    - What could we do instead of using a threshold? 

## 5. Competition: awake / n-rem / rem 

Same experiment as the second one but this time you're free to do any modifications you want 
as long as it's still an MLP. You can change how you validate the model if you want. Just make 
sure to give the important details in your report. 

Join your results on the test data (make sure to preprocess the test data the same way you 
preprocessed your training data before predicting on it). 

The first three groups will have a bonus. 

### What you need to do: 

- Implement at least one new idea. 
- You don't need to apply more than one idea if you don't want to. This competition is there to 
    motivate you, not to burden you with a lot of work. Only the top 3 winners will appear in the 
    ranking, but we will inform you of your score and rank privately. 
- It can be ideas from this list but can also be other ideas you find yourselves. 

### Ideas: 
- Change [optimizer](https://keras.io/api/optimizers/) (e.g. Adam instead of SGD) 
- Change last layer activation function by « softmax » (for each output of the model we will 
    predict a probability and the sum of the output values equal to 1). 
- Use [CategoricalCrossentropy](https://keras.io/api/losses/probabilistic_losses/#categoricalcrossentropy-class) (which is a loss meant for classification, ) instead of « MSE » 
- Use an [ensemble  of models]((https://machinelearningmastery.com/model-averaging-ensemble-for-deep-learning-neural-networks/)) (train multiple models, average their predictions) 

## 6. Submission 

Provide your report, your notebook(s) and your prediction on the competition test set. 
Report must be a PDF. Don't forget to include all your group members names in the report! 

**Competition's predictions:**

Save your results in a file named « test_pred.npy » and add it to your submission. Make sure the 
order of your predictions is correct (don't shuffle test set…). 

test_predictions must be a numpy array with the name of the sleep stages you predicted. 
We cannot know what sleep stage you refer to if you have them as integer! 

Example: `array(['n', 'w', 'r', …])`

```py
with open('test_pred.npy', 'wb') as f: 
    np.save(f, test_predictions)
```